In [1]:
import os
# import jsonlines
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
import re

from dotenv import load_dotenv

tqdm.pandas()
load_dotenv()

True

In [8]:
from huggingface_hub import hf_hub_download, HfFileSystem

In [14]:
hfs = HfFileSystem()
for fpath in tqdm(hfs.listdir('Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', detail=False)):
    fname = fpath.split('/')[-1]
    hf_hub_download(
        repo_id='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256',
        filename=fname,
        local_dir="model"
        )

  0%|          | 0/11 [00:00<?, ?it/s]

In [10]:
ROOT_DATA_DIR = os.path.join("..", "..", "..", *os.getenv("DATA_DIR").split(","))
REQ_FOLDER = os.path.join(ROOT_DATA_DIR, 'InferenceOutput')

os.path.exists(REQ_FOLDER)

True

In [3]:
os.listdir(REQ_FOLDER)

['Round01.jsonl', 'Round02.jsonl', 'Round03.jsonl', 'Round04.jsonl']

In [4]:
final_answers = {}

for idx in tqdm(range(1, 5), desc="Rounds"):
    current_round = f"Round{idx:0>2}"

    df = pd.read_json(os.path.join(REQ_FOLDER, f'{current_round}.jsonl'), lines=True)

    for _, current_row in tqdm(df[df['Final_Answer'].notna()].iterrows(), total=len(df[df['Final_Answer'].notna()]), desc=f"Processing {current_round}"):

        exp_id = current_row['ExpID']
        fa = current_row['Final_Answer']

        if exp_id in final_answers:
            final_answers[exp_id].append(fa)
        else:
            final_answers[exp_id] = [fa]


Rounds:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Round01: 0it [00:00, ?it/s]

Processing Round02:   0%|          | 0/4459 [00:00<?, ?it/s]

Processing Round03:   0%|          | 0/62179 [00:00<?, ?it/s]

Processing Round04:   0%|          | 0/178982 [00:00<?, ?it/s]

In [5]:
len(final_answers)

2494

In [15]:
def get_final_solution(x):
    sols = re.findall(r'\\boxed\{(.*?)\}', x, re.DOTALL)

    if len(sols) == 0:
        sols = re.findall(r'\\boxed (.*?)\$', x, re.DOTALL)
    
    # Check if all are same
    try:
        if all(sol == sols[0] for sol in sols):
            return sols[0]
        else:
            return sols[-1]
    except:
        print(sols)
        print(x)
        raise


In [19]:
MATH_ds = load_dataset('hendrycks/competition_math', trust_remote_code=True)['test']
MATH_df = MATH_ds.to_pandas()

MATH_df['final_answer'] = MATH_df['solution'].progress_apply(get_final_solution)
MATH_df['final_answer_is_numeric'] = MATH_df['final_answer'].progress_apply(lambda x: x.isnumeric())

MATH_numeric_df = MATH_df[MATH_df['final_answer_is_numeric']].copy(deep=True).reset_index(drop=True)
MATH_numeric_df['ExpID'] = [f"Exp{idx:0>4}" for idx in range(len(MATH_numeric_df))]
MATH_numeric_df

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True,Exp0000
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True,Exp0001
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True,Exp0002
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True,Exp0003
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True,Exp0004
...,...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True,Exp2878
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True,Exp2879
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True,Exp2880
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True,Exp2881


In [20]:
MATH_numeric_df[MATH_numeric_df['ExpID'] == 'Exp0695']

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
695,The expression $x^2 + 18x - 63$ can be written...,Level 2,Algebra,"Factoring, we find that $x^2 + 18x - 63 = (x -...",21,True,Exp0695


In [47]:
correct_exps = []
incorrect_exps = []
for exp_id in tqdm(final_answers):

    top3 = [x for x, _ in Counter(final_answers[exp_id]).most_common()[:3]]
    ofa = MATH_numeric_df[MATH_numeric_df['ExpID'] == exp_id].final_answer.values[0]

    ofa = str(ofa) if not isinstance(ofa, str) else ofa

    if ofa in top3:
        correct_exps.append(exp_id)
    else:
        incorrect_exps.append(exp_id)

  0%|          | 0/2494 [00:00<?, ?it/s]

In [60]:
len(correct_exps), len(incorrect_exps)

(135, 2359)

In [56]:
idx = 1
MATH_numeric_df[MATH_numeric_df['ExpID'] == incorrect_exps[idx]]

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
22,"The point $(a, b)$ lies on the line with the e...",Level 2,Algebra,We plug in $x = 4$: \begin{align*}\n3(4) + 2y ...,0,True,Exp0022


In [57]:
ofa = MATH_numeric_df[MATH_numeric_df['ExpID'] == incorrect_exps[idx]].final_answer.values[0]
# ofa, [x for x, _ in Counter(final_answers[incorrect_exps[idx]]).most_common()[:3]], str(ofa) in [x for x, _ in Counter(final_answers[incorrect_exps[idx]]).most_common()[:3]]

Counter(final_answers[incorrect_exps[idx]]).most_common()

[('2', 20),
 ('4', 19),
 ('8/17', 15),
 ('0', 5),
 ('1', 5),
 ('243', 5),
 ('27', 5),
 ('63360', 5),
 ('6/17', 5),
 ('2/3', 5),
 ('-1/15', 5),
 ('17/15', 5),
 ('There is no solution.', 5),
 ('10.54', 5),
 ('There is no solution for e.', 5)]

In [59]:
print(df[df['ExpID'] == incorrect_exps[idx]].Current_Prompt.values[0])

Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>In the sequence 0, 1, 1, 3, 6, 9, 27, ..., the first term is 0.   Subsequent terms are produced by alternately adding and  multiplying by each successive integer beginning with 1.  For  instance, the second term is produced by adding  1 to the first term; the third term

In [77]:
test = """Collecting vllm
  Downloading vllm-0.4.3-cp310-cp310-manylinux1_x86_64.whl.metadata (7.8 kB)
Collecting cmake>=3.21 (from vllm)
  Downloading cmake-3.29.5-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.1 kB)
Requirement already satisfied: ninja in /opt/conda/lib/python3.10/site-packages (from vllm) (1.11.1.1)
Requirement already satisfied: psutil in /opt/conda/lib/python3.10/site-packages (from vllm) (5.9.3)
Requirement already satisfied: sentencepiece in /opt/conda/lib/python3.10/site-packages (from vllm) (0.2.0)
Requirement already satisfied: numpy in /opt/conda/lib/python3.10/site-packages (from vllm) (1.26.4)
Requirement already satisfied: requests in /opt/conda/lib/python3.10/site-packages (from vllm) (2.32.3)
Requirement already satisfied: py-cpuinfo in /opt/conda/lib/python3.10/site-packages (from vllm) (9.0.0)
Requirement already satisfied: transformers>=4.40.0 in /opt/conda/lib/python3.10/site-packages (from vllm) (4.41.2)
Requirement already satisfied: tokenizers>=0.19.1 in /opt/conda/lib/python3.10/site-packages (from vllm) (0.19.1)
Requirement already satisfied: fastapi in /opt/conda/lib/python3.10/site-packages (from vllm) (0.108.0)
Requirement already satisfied: aiohttp in /opt/conda/lib/python3.10/site-packages (from vllm) (3.9.1)
Collecting openai (from vllm)
  Downloading openai-1.33.0-py3-none-any.whl.metadata (21 kB)
Requirement already satisfied: uvicorn[standard] in /opt/conda/lib/python3.10/site-packages (from vllm) (0.25.0)
Requirement already satisfied: pydantic>=2.0 in /opt/conda/lib/python3.10/site-packages (from vllm) (2.5.3)
Requirement already satisfied: prometheus-client>=0.18.0 in /opt/conda/lib/python3.10/site-packages (from vllm) (0.19.0)
Collecting prometheus-fastapi-instrumentator>=7.0.0 (from vllm)
  Downloading prometheus_fastapi_instrumentator-7.0.0-py3-none-any.whl.metadata (13 kB)
Collecting tiktoken>=0.6.0 (from vllm)
  Downloading tiktoken-0.7.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
Collecting lm-format-enforcer==0.10.1 (from vllm)
  Downloading lm_format_enforcer-0.10.1-py3-none-any.whl.metadata (16 kB)
Collecting outlines==0.0.34 (from vllm)
  Downloading outlines-0.0.34-py3-none-any.whl.metadata (13 kB)
Requirement already satisfied: typing-extensions in /opt/conda/lib/python3.10/site-packages (from vllm) (4.9.0)
Requirement already satisfied: filelock>=3.10.4 in /opt/conda/lib/python3.10/site-packages (from vllm) (3.13.1)
Requirement already satisfied: ray>=2.9 in /opt/conda/lib/python3.10/site-packages (from vllm) (2.9.0)
Requirement already satisfied: nvidia-ml-py in /opt/conda/lib/python3.10/site-packages (from vllm) (11.495.46)
Collecting torch==2.3.0 (from vllm)
  Downloading torch-2.3.0-cp310-cp310-manylinux1_x86_64.whl.metadata (26 kB)
Collecting xformers==0.0.26.post1 (from vllm)
  Downloading xformers-0.0.26.post1-cp310-cp310-manylinux2014_x86_64.whl.metadata (1.0 kB)
Collecting vllm-flash-attn==2.5.8.post2 (from vllm)
  Downloading vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl.metadata (482 bytes)
Collecting interegular>=0.3.2 (from lm-format-enforcer==0.10.1->vllm)
  Downloading interegular-0.3.3-py37-none-any.whl.metadata (3.0 kB)
Requirement already satisfied: packaging in /opt/conda/lib/python3.10/site-packages (from lm-format-enforcer==0.10.1->vllm) (21.3)
Requirement already satisfied: pyyaml in /opt/conda/lib/python3.10/site-packages (from lm-format-enforcer==0.10.1->vllm) (6.0.1)
Requirement already satisfied: jinja2 in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (3.1.2)
Collecting lark (from outlines==0.0.34->vllm)
  Downloading lark-1.1.9-py3-none-any.whl.metadata (1.9 kB)
Requirement already satisfied: nest-asyncio in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (1.5.8)
Requirement already satisfied: cloudpickle in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (2.2.1)
Collecting diskcache (from outlines==0.0.34->vllm)
  Downloading diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Requirement already satisfied: scipy in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (1.11.4)
Requirement already satisfied: numba in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (0.58.1)
Requirement already satisfied: joblib in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (1.4.2)
Requirement already satisfied: referencing in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (0.32.1)
Requirement already satisfied: jsonschema in /opt/conda/lib/python3.10/site-packages (from outlines==0.0.34->vllm) (4.20.0)
Requirement already satisfied: sympy in /opt/conda/lib/python3.10/site-packages (from torch==2.3.0->vllm) (1.12.1)
Requirement already satisfied: networkx in /opt/conda/lib/python3.10/site-packages (from torch==2.3.0->vllm) (3.2.1)
Requirement already satisfied: fsspec in /opt/conda/lib/python3.10/site-packages (from torch==2.3.0->vllm) (2024.3.1)
Collecting nvidia-cuda-nvrtc-cu12==12.1.105 (from torch==2.3.0->vllm)
  Downloading nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
Collecting nvidia-cuda-runtime-cu12==12.1.105 (from torch==2.3.0->vllm)
  Downloading nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
Collecting nvidia-cuda-cupti-cu12==12.1.105 (from torch==2.3.0->vllm)
  Downloading nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
Collecting nvidia-cudnn-cu12==8.9.2.26 (from torch==2.3.0->vllm)
  Downloading nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
Collecting nvidia-cublas-cu12==12.1.3.1 (from torch==2.3.0->vllm)
  Downloading nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
Collecting nvidia-cufft-cu12==11.0.2.54 (from torch==2.3.0->vllm)
  Downloading nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
Collecting nvidia-curand-cu12==10.3.2.106 (from torch==2.3.0->vllm)
  Downloading nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
Collecting nvidia-cusolver-cu12==11.4.5.107 (from torch==2.3.0->vllm)
  Downloading nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
Collecting nvidia-cusparse-cu12==12.1.0.106 (from torch==2.3.0->vllm)
  Downloading nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
Collecting nvidia-nccl-cu12==2.20.5 (from torch==2.3.0->vllm)
  Downloading nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
Collecting nvidia-nvtx-cu12==12.1.105 (from torch==2.3.0->vllm)
  Downloading nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.7 kB)
Collecting triton==2.3.0 (from torch==2.3.0->vllm)
  Downloading triton-2.3.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Collecting nvidia-nvjitlink-cu12 (from nvidia-cusolver-cu12==11.4.5.107->torch==2.3.0->vllm)
  Downloading nvidia_nvjitlink_cu12-12.5.40-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
Requirement already satisfied: starlette<1.0.0,>=0.30.0 in /opt/conda/lib/python3.10/site-packages (from prometheus-fastapi-instrumentator>=7.0.0->vllm) (0.32.0.post1)
Requirement already satisfied: annotated-types>=0.4.0 in /opt/conda/lib/python3.10/site-packages (from pydantic>=2.0->vllm) (0.6.0)
Requirement already satisfied: pydantic-core==2.14.6 in /opt/conda/lib/python3.10/site-packages (from pydantic>=2.0->vllm) (2.14.6)
Requirement already satisfied: click>=7.0 in /opt/conda/lib/python3.10/site-packages (from ray>=2.9->vllm) (8.1.7)
Requirement already satisfied: msgpack<2.0.0,>=1.0.0 in /opt/conda/lib/python3.10/site-packages (from ray>=2.9->vllm) (1.0.7)
Requirement already satisfied: protobuf!=3.19.5,>=3.15.3 in /opt/conda/lib/python3.10/site-packages (from ray>=2.9->vllm) (3.20.3)
Requirement already satisfied: aiosignal in /opt/conda/lib/python3.10/site-packages (from ray>=2.9->vllm) (1.3.1)
Requirement already satisfied: frozenlist in /opt/conda/lib/python3.10/site-packages (from ray>=2.9->vllm) (1.4.1)
Requirement already satisfied: regex>=2022.1.18 in /opt/conda/lib/python3.10/site-packages (from tiktoken>=0.6.0->vllm) (2023.12.25)
Requirement already satisfied: charset-normalizer<4,>=2 in /opt/conda/lib/python3.10/site-packages (from requests->vllm) (3.3.2)
Requirement already satisfied: idna<4,>=2.5 in /opt/conda/lib/python3.10/site-packages (from requests->vllm) (3.6)
Requirement already satisfied: urllib3<3,>=1.21.1 in /opt/conda/lib/python3.10/site-packages (from requests->vllm) (1.26.18)
Requirement already satisfied: certifi>=2017.4.17 in /opt/conda/lib/python3.10/site-packages (from requests->vllm) (2024.2.2)
Requirement already satisfied: huggingface-hub<1.0,>=0.16.4 in /opt/conda/lib/python3.10/site-packages (from tokenizers>=0.19.1->vllm) (0.23.2)
Requirement already satisfied: safetensors>=0.4.1 in /opt/conda/lib/python3.10/site-packages (from transformers>=4.40.0->vllm) (0.4.3)
Requirement already satisfied: tqdm>=4.27 in /opt/conda/lib/python3.10/site-packages (from transformers>=4.40.0->vllm) (4.66.4)
Requirement already satisfied: attrs>=17.3.0 in /opt/conda/lib/python3.10/site-packages (from aiohttp->vllm) (23.2.0)
Requirement already satisfied: multidict<7.0,>=4.5 in /opt/conda/lib/python3.10/site-packages (from aiohttp->vllm) (6.0.4)
Requirement already satisfied: yarl<2.0,>=1.0 in /opt/conda/lib/python3.10/site-packages (from aiohttp->vllm) (1.9.3)
Requirement already satisfied: async-timeout<5.0,>=4.0 in /opt/conda/lib/python3.10/site-packages (from aiohttp->vllm) (4.0.3)
Requirement already satisfied: anyio<5,>=3.5.0 in /opt/conda/lib/python3.10/site-packages (from openai->vllm) (4.2.0)
Requirement already satisfied: distro<2,>=1.7.0 in /opt/conda/lib/python3.10/site-packages (from openai->vllm) (1.9.0)
Requirement already satisfied: httpx<1,>=0.23.0 in /opt/conda/lib/python3.10/site-packages (from openai->vllm) (0.27.0)
Requirement already satisfied: sniffio in /opt/conda/lib/python3.10/site-packages (from openai->vllm) (1.3.0)
Requirement already satisfied: h11>=0.8 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (0.14.0)
Requirement already satisfied: httptools>=0.5.0 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (0.6.1)
Requirement already satisfied: python-dotenv>=0.13 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (1.0.0)
Requirement already satisfied: uvloop!=0.15.0,!=0.15.1,>=0.14.0 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (0.19.0)
Requirement already satisfied: watchfiles>=0.13 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (0.21.0)
Requirement already satisfied: websockets>=10.4 in /opt/conda/lib/python3.10/site-packages (from uvicorn[standard]->vllm) (12.0)
Requirement already satisfied: exceptiongroup>=1.0.2 in /opt/conda/lib/python3.10/site-packages (from anyio<5,>=3.5.0->openai->vllm) (1.2.0)
Requirement already satisfied: httpcore==1.* in /opt/conda/lib/python3.10/site-packages (from httpx<1,>=0.23.0->openai->vllm) (1.0.5)
Requirement already satisfied: pyparsing!=3.0.5,>=2.0.2 in /opt/conda/lib/python3.10/site-packages (from packaging->lm-format-enforcer==0.10.1->vllm) (3.1.1)
Requirement already satisfied: MarkupSafe>=2.0 in /opt/conda/lib/python3.10/site-packages (from jinja2->outlines==0.0.34->vllm) (2.1.3)
Requirement already satisfied: jsonschema-specifications>=2023.03.6 in /opt/conda/lib/python3.10/site-packages (from jsonschema->outlines==0.0.34->vllm) (2023.12.1)
Requirement already satisfied: rpds-py>=0.7.1 in /opt/conda/lib/python3.10/site-packages (from jsonschema->outlines==0.0.34->vllm) (0.16.2)
Requirement already satisfied: llvmlite<0.42,>=0.41.0dev0 in /opt/conda/lib/python3.10/site-packages (from numba->outlines==0.0.34->vllm) (0.41.1)
Requirement already satisfied: mpmath<1.4.0,>=1.1.0 in /opt/conda/lib/python3.10/site-packages (from sympy->torch==2.3.0->vllm) (1.3.0)
Downloading vllm-0.4.3-cp310-cp310-manylinux1_x86_64.whl (131.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 4.3 MB/s eta 0:00:0000:0100:01
Downloading lm_format_enforcer-0.10.1-py3-none-any.whl (42 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 3.0 MB/s eta 0:00:00
Downloading outlines-0.0.34-py3-none-any.whl (76 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 4.2 MB/s eta 0:00:00
Downloading torch-2.3.0-cp310-cp310-manylinux1_x86_64.whl (779.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.4 MB/s eta 0:00:00:00:0100:01
Downloading vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl (37.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 16.6 MB/s eta 0:00:0000:0100:01
Downloading xformers-0.0.26.post1-cp310-cp310-manylinux2014_x86_64.whl (222.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.7/222.7 MB 4.7 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.4 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 13.4 MB/s eta 0:00:0000:0100:01
Downloading nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 4.4 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 13.3 MB/s eta 0:00:00a 0:00:01
Downloading nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00:00:0100:02
Downloading nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 4.6 MB/s eta 0:00:0000:0100:01
Downloading nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 7.4 MB/s eta 0:00:0000:0100:01
Downloading nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 9.8 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.6 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 7.8 MB/s eta 0:00:00:00:0100:01
Downloading nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 7.1 MB/s eta 0:00:00
Downloading triton-2.3.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (168.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 6.7 MB/s eta 0:00:00:00:0100:01
Downloading cmake-3.29.5-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 26.1 MB/s eta 0:00:0000:0100:01
Downloading prometheus_fastapi_instrumentator-7.0.0-py3-none-any.whl (19 kB)
Downloading tiktoken-0.7.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.6 MB/s eta 0:00:0000:01
Downloading openai-1.33.0-py3-none-any.whl (325 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 22.4 MB/s eta 0:00:00
Downloading interegular-0.3.3-py37-none-any.whl (23 kB)
Downloading diskcache-5.6.3-py3-none-any.whl (45 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
Downloading lark-1.1.9-py3-none-any.whl (111 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 9.9 MB/s eta 0:00:00
Downloading nvidia_nvjitlink_cu12-12.5.40-py3-none-manylinux2014_x86_64.whl (21.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 27.4 MB/s eta 0:00:0000:0100:01
Installing collected packages: triton, nvidia-nvtx-cu12, nvidia-nvjitlink-cu12, nvidia-nccl-cu12, nvidia-curand-cu12, nvidia-cufft-cu12, nvidia-cuda-runtime-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-cupti-cu12, nvidia-cublas-cu12, lark, interegular, diskcache, cmake, tiktoken, nvidia-cusparse-cu12, nvidia-cudnn-cu12, prometheus-fastapi-instrumentator, openai, nvidia-cusolver-cu12, lm-format-enforcer, torch, xformers, vllm-flash-attn, outlines, vllm
  Attempting uninstall: torch
    Found existing installation: torch 2.1.2
    Uninstalling torch-2.1.2:
      Successfully uninstalled torch-2.1.2
Successfully installed cmake-3.29.5 diskcache-5.6.3 interegular-0.3.3 lark-1.1.9 lm-format-enforcer-0.10.1 nvidia-cublas-cu12-12.1.3.1 nvidia-cuda-cupti-cu12-12.1.105 nvidia-cuda-nvrtc-cu12-12.1.105 nvidia-cuda-runtime-cu12-12.1.105 nvidia-cudnn-cu12-8.9.2.26 nvidia-cufft-cu12-11.0.2.54 nvidia-curand-cu12-10.3.2.106 nvidia-cusolver-cu12-11.4.5.107 nvidia-cusparse-cu12-12.1.0.106 nvidia-nccl-cu12-2.20.5 nvidia-nvjitlink-cu12-12.5.40 nvidia-nvtx-cu12-12.1.105 openai-1.33.0 outlines-0.0.34 prometheus-fastapi-instrumentator-7.0.0 tiktoken-0.7.0 torch-2.3.0 triton-2.3.0 vllm-0.4.3 vllm-flash-attn-2.5.8.post2 xformers-0.0.26.post1
Note: you may need to restart the kernel to use updated packages.
"""

In [ ]:
"ipywidgets>=8"

In [16]:
import re

In [80]:
for x in re.findall(r"Downloading (\S+\.whl)", test, re.DOTALL):
    print(x)
# re.findall(r"Downloading \w+-([\d|\.]+)-", test, re.DOTALL)

vllm-0.4.3-cp310-cp310-manylinux1_x86_64.whl
cmake-3.29.5-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
openai-1.33.0-py3-none-any.whl
prometheus_fastapi_instrumentator-7.0.0-py3-none-any.whl
tiktoken-0.7.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
lm_format_enforcer-0.10.1-py3-none-any.whl
outlines-0.0.34-py3-none-any.whl
torch-2.3.0-cp310-cp310-manylinux1_x86_64.whl
xformers-0.0.26.post1-cp310-cp310-manylinux2014_x86_64.whl
vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl
interegular-0.3.3-py37-none-any.whl
lark-1.1.9-py3-none-any.whl
diskcache-5.6.3-py3-none-any.whl
nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl
nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl
nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl
nvidia_curand_cu1

In [81]:
test = """
vllm-0.4.3-cp310-cp310-manylinux1_x86_64.whl
cmake-3.29.5-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
openai-1.33.0-py3-none-any.whl
prometheus_fastapi_instrumentator-7.0.0-py3-none-any.whl
tiktoken-0.7.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
lm_format_enforcer-0.10.1-py3-none-any.whl
outlines-0.0.34-py3-none-any.whl
torch-2.3.0-cp310-cp310-manylinux1_x86_64.whl
xformers-0.0.26.post1-cp310-cp310-manylinux2014_x86_64.whl
vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl
interegular-0.3.3-py37-none-any.whl
lark-1.1.9-py3-none-any.whl
diskcache-5.6.3-py3-none-any.whl
nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl
nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl
nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl
nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl
nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl
nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl
nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl
nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
triton-2.3.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
nvidia_nvjitlink_cu12-12.5.40-py3-none-manylinux2014_x86_64.whl
vllm-0.4.3-cp310-cp310-manylinux1_x86_64.whl
lm_format_enforcer-0.10.1-py3-none-any.whl
outlines-0.0.34-py3-none-any.whl
torch-2.3.0-cp310-cp310-manylinux1_x86_64.whl
vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl
xformers-0.0.26.post1-cp310-cp310-manylinux2014_x86_64.whl
nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl
nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl
nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl
nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl
nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl
nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl
nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl
nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl
triton-2.3.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
cmake-3.29.5-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
prometheus_fastapi_instrumentator-7.0.0-py3-none-any.whl
tiktoken-0.7.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
openai-1.33.0-py3-none-any.whl
interegular-0.3.3-py37-none-any.whl
diskcache-5.6.3-py3-none-any.whl
lark-1.1.9-py3-none-any.whl
nvidia_nvjitlink_cu12-12.5.40-py3-none-manylinux2014_x86_64.whl"""

In [84]:
# test = test.splitlines()[1:]
libs = set()
for lib in test:
    lib_name = re.findall(r"(\w+)-", lib, re.DOTALL)[0]
    lib_n = re.findall(r"\w+-([\d|\.|post]+)-", lib, re.DOTALL)[0]
    # print(f"{lib_name}=={lib_n}")
    libs.add(f"{lib_name}=={lib_n}")

print("\n".join(libs))

prometheus_fastapi_instrumentator==7.0.0
nvidia_cudnn_cu12==8.9.2.26
vllm_flash_attn==2.5.8.post2
outlines==0.0.34
tiktoken==0.7.0
nvidia_cuda_runtime_cu12==12.1.105
nvidia_nvjitlink_cu12==12.5.40
nvidia_nvtx_cu12==12.1.105
triton==2.3.0
vllm==0.4.3
nvidia_nccl_cu12==2.20.5
xformers==0.0.26.post1
nvidia_cufft_cu12==11.0.2.54
nvidia_cuda_nvrtc_cu12==12.1.105
openai==1.33.0
cmake==3.29.5
nvidia_cusparse_cu12==12.1.0.106
lm_format_enforcer==0.10.1
lark==1.1.9
nvidia_cusolver_cu12==11.4.5.107
torch==2.3.0
nvidia_cuda_cupti_cu12==12.1.105
nvidia_curand_cu12==10.3.2.106
diskcache==5.6.3
nvidia_cublas_cu12==12.1.3.1
interegular==0.3.3


In [ ]:
# test = test.splitlines()[1:]
libs = set()
for lib in test:
    lib_name = re.findall(r"(\w+)-", lib, re.DOTALL)[0]
    lib_n = re.findall(r"\w+-([\d|\.|post]+)-", lib, re.DOTALL)[0]
    # print(f"{lib_name}=={lib_n}")
    libs.add(f"{lib_name}=={lib_n}")

print("\n".join(libs))

prometheus_fastapi_instrumentator==7.0.0
nvidia_cudnn_cu12==8.9.2.26
vllm_flash_attn==2.5.8.post2
outlines==0.0.34
tiktoken==0.7.0
nvidia_cuda_runtime_cu12==12.1.105
nvidia_nvjitlink_cu12==12.5.40
nvidia_nvtx_cu12==12.1.105
triton==2.3.0
vllm==0.4.3
nvidia_nccl_cu12==2.20.5
xformers==0.0.26.post1
nvidia_cufft_cu12==11.0.2.54
nvidia_cuda_nvrtc_cu12==12.1.105
openai==1.33.0
cmake==3.29.5
nvidia_cusparse_cu12==12.1.0.106
lm_format_enforcer==0.10.1
lark==1.1.9
nvidia_cusolver_cu12==11.4.5.107
torch==2.3.0
nvidia_cuda_cupti_cu12==12.1.105
nvidia_curand_cu12==10.3.2.106
diskcache==5.6.3
nvidia_cublas_cu12==12.1.3.1
interegular==0.3.3


In [74]:
print(lib)
re.findall(r"\w+-([\d|\.|post]+)-", lib, re.DOTALL)

vllm_flash_attn-2.5.8.post2-cp310-cp310-manylinux1_x86_64.whl


['2.5.8.post2']